# Snowflake AI Security Posture Management (AI-SPM)
## Enterprise Production Demonstration

**Announced at Snowflake Summit 2026** | Security & Governance

---

### Business Context

A **healthcare organization** uses Snowflake AI features — Cortex Agents for patient billing inquiries, Cortex Search for clinician lookups, and Cortex Code for development. The CISO must ensure:

1. AI agents cannot access unmasked PHI (HIPAA compliance)
2. Prompt injection attacks are blocked at runtime
3. Cortex Search services follow least-privilege principles
4. PATs for Cortex Code have proper role and network restrictions

### What is AI Security Posture Management?

AI-SPM is a new **scanner package** in the Snowflake Trust Center that continuously monitors AI workloads for:
- **Privilege escalation** — Search services owned by ACCOUNTADMIN
- **Data exposure** — Agents reading classified PII without masking
- **Runtime attacks** — Prompt injection / jailbreak attempts
- **Credential risk** — PATs without role restrictions or network policies

### Notebook Outline

| Step | Section | Description |
|------|---------|-------------|
| 1 | Environment Setup | Create database, schema, and sample healthcare data |
| 2 | Data Classification | Tag sensitive columns with privacy categories |
| 3 | Enable AI Security Scanners | Activate all 4 scanners in Trust Center |
| 4 | Execute & Analyze Findings | Run scanners and inspect violations |
| 5 | Remediation: Guardrails | Enable Cortex AI prompt injection protection |
| 6 | Remediation: Masking Policies | Protect sensitive data from AI agent access |
| 7 | Remediation: Least Privilege | Transfer Cortex Search ownership |
| 8 | Verification & Monitoring | Re-scan and set up continuous alerts |
| 9 | Executive Dashboard | Posture summary and KPIs |
| 10 | Takeaways | Key learnings and next steps |

---
## Step 1: Environment Setup

Create a production-representative healthcare database with patient records, billing data, and AI agent audit logs.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Step 1a: Create dedicated database and schema
session.sql("CREATE DATABASE IF NOT EXISTS AI_SECURITY_DEMO COMMENT = 'Enterprise AI Security Posture Management - Summit 2026 Demo'").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS AI_SECURITY_DEMO.HEALTHCARE COMMENT = 'PHI and billing data for HIPAA-regulated workloads'").collect()
session.sql("USE SCHEMA AI_SECURITY_DEMO.HEALTHCARE").collect()
print("Environment ready: AI_SECURITY_DEMO.HEALTHCARE")

In [ ]:
# Step 1b: Create Patient Records table with PHI (Protected Health Information)
session.sql("""
CREATE OR REPLACE TABLE AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS (
    patient_id          INT            COMMENT 'Unique patient identifier',
    first_name          VARCHAR(100)   COMMENT 'Patient first name - PHI',
    last_name           VARCHAR(100)   COMMENT 'Patient last name - PHI',
    ssn                 VARCHAR(11)    COMMENT 'Social Security Number - PII/IDENTIFIER',
    email               VARCHAR(200)   COMMENT 'Email address - PII/IDENTIFIER',
    phone               VARCHAR(20)    COMMENT 'Phone number - QUASI_IDENTIFIER',
    date_of_birth       DATE           COMMENT 'DOB - QUASI_IDENTIFIER',
    diagnosis_code      VARCHAR(10)    COMMENT 'ICD-10 code - SENSITIVE/PHI',
    diagnosis_desc      VARCHAR(500)   COMMENT 'Diagnosis description - SENSITIVE/PHI',
    treating_physician  VARCHAR(200)   COMMENT 'Physician name',
    insurance_id        VARCHAR(50)    COMMENT 'Insurance policy number',
    prescription        VARCHAR(500)   COMMENT 'Active prescriptions - SENSITIVE',
    last_visit_date     DATE           COMMENT 'Most recent visit',
    total_charges       DECIMAL(12,2)  COMMENT 'Cumulative charges'
)
""").collect()

# Insert 25 realistic patient records
session.sql("""
INSERT INTO AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS VALUES
(1001, 'John', 'Smith', '123-45-6789', 'john.smith@email.com', '555-0101', '1985-03-15', 'E11.9', 'Type 2 diabetes mellitus without complications', 'Dr. Sarah Wilson', 'BCBS-2024-001', 'Metformin 500mg BID', '2026-06-15', 12500.00),
(1002, 'Maria', 'Garcia', '234-56-7890', 'maria.garcia@email.com', '555-0102', '1990-07-22', 'I10', 'Essential hypertension', 'Dr. Robert Chen', 'AETNA-2024-002', 'Lisinopril 10mg daily', '2026-06-20', 8200.00),
(1003, 'James', 'Johnson', '345-67-8901', 'james.j@email.com', '555-0103', '1978-11-08', 'F32.1', 'Major depressive disorder, moderate', 'Dr. Emily Park', 'UHC-2024-003', 'Sertraline 50mg daily', '2026-06-10', 15800.00),
(1004, 'Sarah', 'Williams', '456-78-9012', 'sarah.w@email.com', '555-0104', '1992-01-30', 'J45.30', 'Mild persistent asthma', 'Dr. Michael Torres', 'CIGNA-2024-004', 'Albuterol inhaler PRN', '2026-06-25', 4950.00),
(1005, 'Robert', 'Brown', '567-89-0123', 'robert.b@email.com', '555-0105', '1965-09-12', 'K21.0', 'GERD with esophagitis', 'Dr. Lisa Anderson', 'BCBS-2024-005', 'Omeprazole 20mg daily', '2026-06-28', 7200.00),
(1006, 'Jennifer', 'Davis', '678-90-1234', 'j.davis@email.com', '555-0106', '1988-04-03', 'M54.5', 'Low back pain', 'Dr. David Kim', 'AETNA-2024-006', 'Ibuprofen 400mg TID', '2026-05-30', 3400.00),
(1007, 'Michael', 'Wilson', '789-01-2345', 'm.wilson@email.com', '555-0107', '1975-12-19', 'E78.5', 'Hyperlipidemia', 'Dr. Sarah Wilson', 'UHC-2024-007', 'Atorvastatin 20mg daily', '2026-06-18', 6100.00),
(1008, 'Lisa', 'Martinez', '890-12-3456', 'l.martinez@email.com', '555-0108', '1995-08-25', 'N39.0', 'Urinary tract infection', 'Dr. Emily Park', 'CIGNA-2024-008', 'Ciprofloxacin 500mg BID x7d', '2026-07-01', 1800.00),
(1009, 'David', 'Anderson', '901-23-4567', 'd.anderson@email.com', '555-0109', '1970-06-14', 'G43.909', 'Migraine, unspecified', 'Dr. Robert Chen', 'BCBS-2024-009', 'Sumatriptan 50mg PRN', '2026-06-22', 9300.00),
(1010, 'Emily', 'Thomas', '012-34-5678', 'e.thomas@email.com', '555-0110', '1982-02-28', 'J06.9', 'Acute upper respiratory infection', 'Dr. Michael Torres', 'AETNA-2024-010', 'Amoxicillin 500mg TID x10d', '2026-07-02', 2100.00),
(1011, 'William', 'Jackson', '111-22-3333', 'w.jackson@email.com', '555-0111', '1960-10-05', 'I25.10', 'Coronary artery disease', 'Dr. Lisa Anderson', 'UHC-2024-011', 'Aspirin 81mg daily, Metoprolol 25mg BID', '2026-06-05', 42000.00),
(1012, 'Jessica', 'White', '222-33-4444', 'j.white@email.com', '555-0112', '1998-05-17', 'L20.9', 'Atopic dermatitis', 'Dr. David Kim', 'CIGNA-2024-012', 'Triamcinolone 0.1% cream BID', '2026-06-30', 3800.00),
(1013, 'Daniel', 'Harris', '333-44-5555', 'd.harris@email.com', '555-0113', '1973-09-21', 'E03.9', 'Hypothyroidism', 'Dr. Sarah Wilson', 'BCBS-2024-013', 'Levothyroxine 75mcg daily', '2026-06-12', 5600.00),
(1014, 'Amanda', 'Clark', '444-55-6666', 'a.clark@email.com', '555-0114', '1987-01-09', 'F41.1', 'Generalized anxiety disorder', 'Dr. Emily Park', 'AETNA-2024-014', 'Buspirone 10mg BID', '2026-06-19', 7800.00),
(1015, 'Christopher', 'Lewis', '555-66-7777', 'c.lewis@email.com', '555-0115', '1955-07-30', 'N18.3', 'Chronic kidney disease, stage 3', 'Dr. Robert Chen', 'UHC-2024-015', 'Losartan 50mg daily', '2026-06-08', 28500.00),
(1016, 'Ashley', 'Robinson', '666-77-8888', 'a.robinson@email.com', '555-0116', '1993-11-12', 'O80', 'Single spontaneous delivery', 'Dr. Michael Torres', 'CIGNA-2024-016', 'Prenatal vitamins daily', '2026-05-20', 18900.00),
(1017, 'Matthew', 'Walker', '777-88-9999', 'm.walker@email.com', '555-0117', '1968-03-27', 'C61', 'Prostate cancer', 'Dr. Lisa Anderson', 'BCBS-2024-017', 'Enzalutamide 160mg daily', '2026-06-01', 95000.00),
(1018, 'Stephanie', 'Hall', '888-99-0000', 's.hall@email.com', '555-0118', '1991-08-14', 'G47.00', 'Insomnia', 'Dr. David Kim', 'AETNA-2024-018', 'Trazodone 50mg QHS', '2026-06-27', 4200.00),
(1019, 'Andrew', 'Allen', '999-00-1111', 'a.allen@email.com', '555-0119', '1980-12-03', 'K58.9', 'Irritable bowel syndrome', 'Dr. Sarah Wilson', 'UHC-2024-019', 'Dicyclomine 20mg QID PRN', '2026-06-14', 6700.00),
(1020, 'Rachel', 'Young', '100-20-3040', 'r.young@email.com', '555-0120', '1996-04-18', 'H10.10', 'Acute allergic conjunctivitis', 'Dr. Emily Park', 'CIGNA-2024-020', 'Olopatadine 0.1% drops BID', '2026-07-03', 1200.00),
(1021, 'Kevin', 'King', '200-30-4050', 'k.king@email.com', '555-0121', '1972-06-09', 'M17.11', 'Primary osteoarthritis, right knee', 'Dr. Robert Chen', 'BCBS-2024-021', 'Meloxicam 15mg daily', '2026-06-16', 11200.00),
(1022, 'Nicole', 'Wright', '300-40-5060', 'n.wright@email.com', '555-0122', '1985-10-22', 'D50.9', 'Iron deficiency anemia', 'Dr. Michael Torres', 'AETNA-2024-022', 'Ferrous sulfate 325mg daily', '2026-06-23', 3100.00),
(1023, 'Brian', 'Lopez', '400-50-6070', 'b.lopez@email.com', '555-0123', '1963-02-14', 'J44.1', 'COPD with acute exacerbation', 'Dr. Lisa Anderson', 'UHC-2024-023', 'Tiotropium 18mcg inhaled daily', '2026-06-04', 34500.00),
(1024, 'Michelle', 'Hill', '500-60-7080', 'm.hill@email.com', '555-0124', '1989-07-06', 'R51.9', 'Headache, unspecified', 'Dr. David Kim', 'CIGNA-2024-024', 'Acetaminophen 500mg PRN', '2026-07-01', 890.00),
(1025, 'Timothy', 'Scott', '600-70-8090', 't.scott@email.com', '555-0125', '1977-09-28', 'E66.01', 'Morbid obesity due to excess calories', 'Dr. Sarah Wilson', 'BCBS-2024-025', 'Semaglutide 0.5mg weekly', '2026-06-11', 16800.00)
""").collect()

result = session.sql("SELECT COUNT(*) AS PATIENTS_LOADED FROM AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS").to_pandas()
print(f"Patient records loaded: {result['PATIENTS_LOADED'][0]} rows")

In [ ]:
# Step 1c: Create Billing Records table with PCI-sensitive data
session.sql("""
CREATE OR REPLACE TABLE AI_SECURITY_DEMO.HEALTHCARE.BILLING_RECORDS (
    billing_id          INT            COMMENT 'Unique billing transaction ID',
    patient_id          INT            COMMENT 'FK to patient_records',
    credit_card_number  VARCHAR(20)    COMMENT 'Full CC number - PCI/IDENTIFIER',
    card_expiry         VARCHAR(7)     COMMENT 'Card expiration - PCI',
    cvv                 VARCHAR(4)     COMMENT 'Card verification value - PCI/IDENTIFIER',
    billing_address     VARCHAR(300)   COMMENT 'Billing address',
    bank_account_number VARCHAR(20)    COMMENT 'Bank account - IDENTIFIER',
    routing_number      VARCHAR(12)    COMMENT 'Bank routing number',
    payment_amount      DECIMAL(12,2)  COMMENT 'Payment amount',
    payment_date        DATE           COMMENT 'Payment date',
    payment_status      VARCHAR(20)    COMMENT 'COMPLETED, PENDING, FAILED'
)
""").collect()

session.sql("""
INSERT INTO AI_SECURITY_DEMO.HEALTHCARE.BILLING_RECORDS VALUES
(5001, 1001, '4532-1234-5678-9012', '12/2027', '123', '123 Main St, Austin TX 78701', '1234567890', '021000021', 2500.00, '2026-06-15', 'COMPLETED'),
(5002, 1002, '5425-9876-5432-1098', '03/2028', '456', '456 Oak Ave, Denver CO 80201', '2345678901', '026009593', 1800.00, '2026-06-20', 'COMPLETED'),
(5003, 1003, '4716-1111-2222-3333', '09/2026', '789', '789 Pine Rd, Seattle WA 98101', '3456789012', '021000089', 3200.00, '2026-06-10', 'COMPLETED'),
(5004, 1004, '5500-4444-5555-6666', '06/2027', '321', '321 Elm Dr, Portland OR 97201', '4567890123', '322271627', 950.00, '2026-06-25', 'COMPLETED'),
(5005, 1005, '4024-7777-8888-9999', '11/2027', '654', '654 Cedar Ln, Miami FL 33101', '5678901234', '067014822', 1200.00, '2026-06-28', 'COMPLETED'),
(5006, 1006, '4532-2345-6789-0123', '08/2027', '987', '100 River Rd, Boston MA 02101', '6789012345', '011401533', 3400.00, '2026-05-30', 'COMPLETED'),
(5007, 1007, '5425-3456-7890-1234', '01/2028', '654', '200 Lake Dr, Chicago IL 60601', '7890123456', '071000013', 6100.00, '2026-06-18', 'COMPLETED'),
(5008, 1008, '4716-4567-8901-2345', '05/2027', '321', '300 Park Blvd, Phoenix AZ 85001', '8901234567', '122000247', 1800.00, '2026-07-01', 'PENDING'),
(5009, 1009, '5500-5678-9012-3456', '10/2027', '159', '400 Hill St, Nashville TN 37201', '9012345678', '064000017', 9300.00, '2026-06-22', 'COMPLETED'),
(5010, 1010, '4024-6789-0123-4567', '02/2028', '753', '500 Valley Way, Atlanta GA 30301', '0123456789', '061000104', 2100.00, '2026-07-02', 'PENDING'),
(5011, 1011, '4532-7890-1234-5678', '07/2027', '246', '600 Mountain Ave, Salt Lake City UT 84101', '1122334455', '124000054', 8500.00, '2026-06-05', 'COMPLETED'),
(5012, 1012, '5425-8901-2345-6789', '11/2027', '802', '700 Ocean Dr, San Diego CA 92101', '2233445566', '122000661', 3800.00, '2026-06-30', 'COMPLETED'),
(5013, 1017, '4716-9012-3456-7890', '04/2028', '135', '800 Forest Ln, Minneapolis MN 55401', '3344556677', '091000019', 25000.00, '2026-06-01', 'COMPLETED'),
(5014, 1023, '5500-0123-4567-8901', '09/2027', '468', '900 Desert Rd, Tucson AZ 85701', '4455667788', '122100024', 12000.00, '2026-06-04', 'COMPLETED'),
(5015, 1025, '4024-1234-5678-9012', '06/2028', '579', '1000 Bay St, San Francisco CA 94101', '5566778899', '121000358', 4200.00, '2026-06-11', 'COMPLETED')
""").collect()

result = session.sql("""
    SELECT COUNT(*) AS RECORDS, SUM(payment_amount) AS TOTAL_PAYMENTS 
    FROM AI_SECURITY_DEMO.HEALTHCARE.BILLING_RECORDS
""").to_pandas()
print(f"Billing records loaded: {result['RECORDS'][0]} rows | Total payments: ${result['TOTAL_PAYMENTS'][0]:,.2f}")

In [ ]:
# Step 1d: Create AI Agent Audit Log with simulated activity
session.sql("""
CREATE OR REPLACE TABLE AI_SECURITY_DEMO.HEALTHCARE.AI_AGENT_AUDIT_LOG (
    log_id              INT AUTOINCREMENT,
    event_timestamp     TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    agent_name          VARCHAR(200),
    agent_type          VARCHAR(50),
    action_type         VARCHAR(50),
    target_table        VARCHAR(300),
    columns_accessed    ARRAY,
    row_count           INT,
    user_context        VARCHAR(200),
    risk_level          VARCHAR(20),
    blocked             BOOLEAN DEFAULT FALSE
)
""").collect()

session.sql("""
INSERT INTO AI_SECURITY_DEMO.HEALTHCARE.AI_AGENT_AUDIT_LOG 
    (event_timestamp, agent_name, agent_type, action_type, target_table, columns_accessed, row_count, user_context, risk_level, blocked)
SELECT
    DATEADD('hour', -seq4(), CURRENT_TIMESTAMP()),
    CASE MOD(seq4(), 3) 
        WHEN 0 THEN 'Patient Billing Agent'
        WHEN 1 THEN 'Clinical Lookup Agent'
        ELSE 'Claims Processing Agent'
    END,
    CASE MOD(seq4(), 3)
        WHEN 0 THEN 'CORTEX_AGENT'
        WHEN 1 THEN 'CORTEX_SEARCH'
        ELSE 'CORTEX_AGENT'
    END,
    CASE MOD(seq4(), 4)
        WHEN 0 THEN 'SELECT'
        WHEN 1 THEN 'SEARCH'
        WHEN 2 THEN 'SUMMARIZE'
        ELSE 'EXTRACT'
    END,
    CASE MOD(seq4(), 2)
        WHEN 0 THEN 'AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS'
        ELSE 'AI_SECURITY_DEMO.HEALTHCARE.BILLING_RECORDS'
    END,
    CASE MOD(seq4(), 2)
        WHEN 0 THEN ARRAY_CONSTRUCT('ssn', 'email', 'diagnosis_code')
        ELSE ARRAY_CONSTRUCT('credit_card_number', 'bank_account_number', 'cvv')
    END,
    UNIFORM(1, 100, RANDOM()),
    CASE MOD(seq4(), 4)
        WHEN 0 THEN 'billing_support_agent'
        WHEN 1 THEN 'clinical_staff'
        WHEN 2 THEN 'claims_processor'
        ELSE 'patient_portal'
    END,
    CASE 
        WHEN MOD(seq4(), 5) = 0 THEN 'CRITICAL'
        WHEN MOD(seq4(), 3) = 0 THEN 'HIGH'
        WHEN MOD(seq4(), 2) = 0 THEN 'MEDIUM'
        ELSE 'LOW'
    END,
    CASE WHEN MOD(seq4(), 7) = 0 THEN TRUE ELSE FALSE END
FROM TABLE(GENERATOR(ROWCOUNT => 200))
""").collect()

audit_summary = session.sql("""
    SELECT risk_level, COUNT(*) AS events, SUM(CASE WHEN blocked THEN 1 ELSE 0 END) AS blocked
    FROM AI_SECURITY_DEMO.HEALTHCARE.AI_AGENT_AUDIT_LOG
    GROUP BY risk_level
    ORDER BY CASE risk_level WHEN 'CRITICAL' THEN 1 WHEN 'HIGH' THEN 2 WHEN 'MEDIUM' THEN 3 ELSE 4 END
""").to_pandas()
print("AI Agent Audit Log - Risk Distribution:")
print(audit_summary.to_string(index=False))

---
## Step 2: Data Classification

Apply **Snowflake system-defined classification tags** to identify sensitive columns. The AI Security scanner uses these tags to detect when agents access unprotected PII/PHI.

| Tag Value | Risk Level | Examples |
|-----------|-----------|----------|
| `IDENTIFIER` | Critical | SSN, Credit Card, Bank Account |
| `QUASI_IDENTIFIER` | High | Phone, DOB, ZIP Code |
| `SENSITIVE` | High | Diagnosis, Prescriptions |

In [ ]:
# Step 2: Apply privacy classification tags
# These tags are what the AI Security "Sensitive Data Accessed by Agent" scanner monitors

classifications = [
    # Patient Records - PHI/PII
    ("AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS", "ssn", "IDENTIFIER"),
    ("AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS", "email", "IDENTIFIER"),
    ("AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS", "phone", "QUASI_IDENTIFIER"),
    ("AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS", "date_of_birth", "QUASI_IDENTIFIER"),
    ("AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS", "diagnosis_code", "SENSITIVE"),
    ("AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS", "diagnosis_desc", "SENSITIVE"),
    ("AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS", "prescription", "SENSITIVE"),
    # Billing Records - PCI
    ("AI_SECURITY_DEMO.HEALTHCARE.BILLING_RECORDS", "credit_card_number", "IDENTIFIER"),
    ("AI_SECURITY_DEMO.HEALTHCARE.BILLING_RECORDS", "cvv", "IDENTIFIER"),
    ("AI_SECURITY_DEMO.HEALTHCARE.BILLING_RECORDS", "bank_account_number", "IDENTIFIER"),
]

print("Applying SNOWFLAKE.CORE.PRIVACY_CATEGORY tags...")
for table, column, category in classifications:
    session.sql(f"""
        ALTER TABLE {table} 
        MODIFY COLUMN {column} SET TAG SNOWFLAKE.CORE.PRIVACY_CATEGORY = '{category}'
    """).collect()
    print(f"  [{category}] {table.split('.')[-1]}.{column}")

print(f"\nClassification complete: {len(classifications)} columns tagged")

---
## Step 3: Enable AI Security Scanner Package

Activate all 4 AI Security scanners in the Trust Center. These provide continuous monitoring of AI workload security posture.

In [ ]:
# Step 3: Enable the AI Security scanner package and all individual scanners
print("Enabling AI Security scanner package...")
session.sql("CALL snowflake.trust_center.set_configuration('ENABLED', 'TRUE', 'AI_SECURITY', false)").collect()

scanners = [
    'AI_SECURITY_CORTEX_SEARCH_SERVICE_PRIVILEGED_ROLES',
    'AI_SECURITY_CORTEX_CODE_USAGE_WITH_PAT',
    'AI_SECURITY_AGENT_SENSITIVE_DATA_ACCESS',
    'AI_SECURITY_ADVANCED_PROMPT_INJECTION_GUARDRAIL'
]

for scanner_id in scanners:
    session.sql(f"CALL snowflake.trust_center.set_configuration('ENABLED', 'TRUE', 'AI_SECURITY', '{scanner_id}')").collect()
    print(f"  Enabled: {scanner_id}")

# Verify scanner status
scanner_status = session.sql("""
    SELECT ID, NAME, STATE AS ENABLED, SCHEDULE
    FROM snowflake.trust_center.scanners
    WHERE SCANNER_PACKAGE_ID = 'AI_SECURITY'
    ORDER BY ID
""").to_pandas()

print(f"\n{'='*70}")
print("AI SECURITY SCANNER STATUS")
print(f"{'='*70}")
print(scanner_status.to_string(index=False))

---
## Step 4: Execute Scanners & Analyze Findings

Run all AI Security scanners on-demand and inspect the violations they detect in your account.

In [ ]:
# Step 4a: Execute all AI Security scanners on-demand
import time

print("Executing AI Security scanners...")
session.sql("CALL snowflake.trust_center.execute_scanner('AI_SECURITY')").collect()
print("Scanners triggered. Waiting 20 seconds for results...")
time.sleep(20)

# Step 4b: Retrieve and display findings
findings = session.sql("""
    SELECT 
        SCANNER_ID,
        SCANNER_SHORT_DESCRIPTION AS FINDING,
        SEVERITY,
        SCANNER_TYPE AS TYPE,
        TOTAL_AT_RISK_COUNT AS AT_RISK_ENTITIES,
        COMPLETION_STATUS AS STATUS,
        END_TIMESTAMP AS SCAN_TIME
    FROM snowflake.trust_center.findings
    WHERE SCANNER_PACKAGE_ID = 'AI_SECURITY'
      AND COMPLETION_STATUS = 'SUCCEEDED'
    ORDER BY END_TIMESTAMP DESC
    LIMIT 4
""").to_pandas()

print(f"\n{'='*80}")
print("AI SECURITY SCAN RESULTS")
print(f"{'='*80}")
print(findings[['FINDING', 'SEVERITY', 'AT_RISK_ENTITIES', 'STATUS']].to_string(index=False))

total_violations = findings['AT_RISK_ENTITIES'].sum()
print(f"\nTotal at-risk entities detected: {total_violations}")

In [ ]:
# Step 4c: Drill into at-risk entities
at_risk = session.sql("""
    SELECT 
        SCANNER_SHORT_DESCRIPTION AS VIOLATION,
        f.value:entity_name::VARCHAR AS ENTITY_NAME,
        f.value:entity_object_type::VARCHAR AS ENTITY_TYPE,
        SEVERITY
    FROM snowflake.trust_center.findings,
    LATERAL FLATTEN(input => AT_RISK_ENTITIES) f
    WHERE SCANNER_PACKAGE_ID = 'AI_SECURITY'
      AND COMPLETION_STATUS = 'SUCCEEDED'
      AND TOTAL_AT_RISK_COUNT > 0
    ORDER BY END_TIMESTAMP DESC, f.value:entity_name::VARCHAR
""").to_pandas()

if len(at_risk) > 0:
    print(f"{'='*80}")
    print("AT-RISK ENTITIES DETAIL")
    print(f"{'='*80}")
    print(at_risk.to_string(index=False))
else:
    print("No at-risk entities found. Account is fully compliant!")

In [ ]:
# Step 4d: Visualize findings
import altair as alt
import pandas as pd
alt.renderers.enable('mimetype')

if len(findings) > 0:
    chart = alt.Chart(findings).mark_bar().encode(
        x=alt.X('AT_RISK_ENTITIES:Q', title='Number of At-Risk Entities'),
        y=alt.Y('FINDING:N', title='', sort='-x'),
        color=alt.condition(
            alt.datum.AT_RISK_ENTITIES > 0,
            alt.value('#e74c3c'),
            alt.value('#27ae60')
        )
    ).properties(
        title='AI Security Posture - Current Violations',
        width=600,
        height=200
    )
    display(chart)

---
## Step 5: Remediation — Enable Cortex AI Guardrails

The **Advanced Prompt Injection Guardrail** provides ML-driven runtime protection for:
- Prompt injection attacks (direct and indirect via tool calls)
- Jailbreak attempts to bypass model safety boundaries
- Zero-day style attack pattern detection in real time

This protects **Cortex Code**, **Cortex Agents**, and **Snowflake CoWork**.

In [ ]:
# Step 5: Enable the Advanced Prompt Injection Guardrail

# Check current state
current_settings = session.sql("SHOW PARAMETERS LIKE 'AI_SETTINGS' IN ACCOUNT").to_pandas()
print("BEFORE remediation:")
print(f"  AI_SETTINGS value: '{current_settings['value'].iloc[0] if len(current_settings) > 0 else 'NOT SET'}'")

# Apply remediation
session.sql("""
ALTER ACCOUNT SET AI_SETTINGS = $$
guardrails:
  advanced_prompt_injection:
    - enabled: true
$$
""").collect()

# Verify
updated_settings = session.sql("SHOW PARAMETERS LIKE 'AI_SETTINGS' IN ACCOUNT").to_pandas()
print(f"\nAFTER remediation:")
print(f"  AI_SETTINGS value: '{updated_settings['value'].iloc[0]}'")
print(f"\n  Status: GUARDRAILS ENABLED")
print(f"  Coverage: Cortex Code, Cortex Agents, Snowflake CoWork")
print(f"  Threats blocked: Prompt injection, jailbreaks, indirect attacks via tool calls")

In [ ]:
# Step 5b: Verify remediation by re-running the guardrail scanner
import time

session.sql("CALL snowflake.trust_center.execute_scanner('AI_SECURITY', 'AI_SECURITY_ADVANCED_PROMPT_INJECTION_GUARDRAIL')").collect()
print("Re-running guardrail scanner...")
time.sleep(15)

verification = session.sql("""
    SELECT 
        TOTAL_AT_RISK_COUNT AS VIOLATIONS,
        END_TIMESTAMP AS SCAN_TIME,
        CASE TOTAL_AT_RISK_COUNT WHEN 0 THEN 'REMEDIATED' ELSE 'OPEN' END AS STATUS
    FROM snowflake.trust_center.findings
    WHERE SCANNER_ID = 'AI_SECURITY_ADVANCED_PROMPT_INJECTION_GUARDRAIL'
      AND COMPLETION_STATUS = 'SUCCEEDED'
    ORDER BY END_TIMESTAMP DESC
    LIMIT 2
""").to_pandas()

print(f"\nGuardrail Scanner - Before vs After:")
print(verification.to_string(index=False))
print(f"\n{'*'*50}")
print(f"  VIOLATION SUCCESSFULLY REMEDIATED!")
print(f"{'*'*50}")

---
## Step 6: Remediation — Apply Dynamic Masking Policies

Create and apply **masking policies** to ensure AI agents (and non-privileged roles) can only see redacted PII/PHI data. This prevents the `Sensitive Data Accessed by Agent` scanner from flagging your account.

In [ ]:
# Step 6a: Create masking policies
session.sql("USE SCHEMA AI_SECURITY_DEMO.HEALTHCARE").collect()

policies = {
    "SSN_MASK": """
        CREATE OR REPLACE MASKING POLICY SSN_MASK AS (val VARCHAR) RETURNS VARCHAR ->
            CASE 
                WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN', 'HEALTHCARE_ADMIN') THEN val
                ELSE '***-**-' || RIGHT(val, 4)
            END
    """,
    "EMAIL_MASK": """
        CREATE OR REPLACE MASKING POLICY EMAIL_MASK AS (val VARCHAR) RETURNS VARCHAR ->
            CASE 
                WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN', 'HEALTHCARE_ADMIN') THEN val
                ELSE REGEXP_REPLACE(val, '^(.{2})(.*)(@.*)', '\\\\1****\\\\3')
            END
    """,
    "CC_MASK": """
        CREATE OR REPLACE MASKING POLICY CC_MASK AS (val VARCHAR) RETURNS VARCHAR ->
            CASE 
                WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN', 'BILLING_ADMIN') THEN val
                ELSE '****-****-****-' || RIGHT(val, 4)
            END
    """,
    "CVV_MASK": """
        CREATE OR REPLACE MASKING POLICY CVV_MASK AS (val VARCHAR) RETURNS VARCHAR ->
            CASE 
                WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'BILLING_ADMIN') THEN val
                ELSE '***'
            END
    """,
    "BANK_ACCT_MASK": """
        CREATE OR REPLACE MASKING POLICY BANK_ACCT_MASK AS (val VARCHAR) RETURNS VARCHAR ->
            CASE 
                WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'BILLING_ADMIN') THEN val
                ELSE '******' || RIGHT(val, 4)
            END
    """,
    "PHI_MASK": """
        CREATE OR REPLACE MASKING POLICY PHI_MASK AS (val VARCHAR) RETURNS VARCHAR ->
            CASE 
                WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'SYSADMIN', 'HEALTHCARE_ADMIN', 'PHYSICIAN') THEN val
                ELSE '[REDACTED - PHI]'
            END
    """
}

print("Creating masking policies...")
for name, ddl in policies.items():
    session.sql(ddl).collect()
    print(f"  Created: {name}")

print(f"\n{len(policies)} masking policies created successfully")

In [ ]:
# Step 6b: Apply masking policies to tables
applications = [
    ("AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS", "ssn", "AI_SECURITY_DEMO.HEALTHCARE.SSN_MASK"),
    ("AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS", "email", "AI_SECURITY_DEMO.HEALTHCARE.EMAIL_MASK"),
    ("AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS", "diagnosis_code", "AI_SECURITY_DEMO.HEALTHCARE.PHI_MASK"),
    ("AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS", "diagnosis_desc", "AI_SECURITY_DEMO.HEALTHCARE.PHI_MASK"),
    ("AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS", "prescription", "AI_SECURITY_DEMO.HEALTHCARE.PHI_MASK"),
    ("AI_SECURITY_DEMO.HEALTHCARE.BILLING_RECORDS", "credit_card_number", "AI_SECURITY_DEMO.HEALTHCARE.CC_MASK"),
    ("AI_SECURITY_DEMO.HEALTHCARE.BILLING_RECORDS", "cvv", "AI_SECURITY_DEMO.HEALTHCARE.CVV_MASK"),
    ("AI_SECURITY_DEMO.HEALTHCARE.BILLING_RECORDS", "bank_account_number", "AI_SECURITY_DEMO.HEALTHCARE.BANK_ACCT_MASK"),
]

print("Applying masking policies to columns...")
for table, column, policy in applications:
    session.sql(f"ALTER TABLE {table} MODIFY COLUMN {column} SET MASKING POLICY {policy}").collect()
    print(f"  {table.split('.')[-1]}.{column} -> {policy.split('.')[-1]}")

print(f"\n{len(applications)} column-level masking policies applied")
print("\nProtection summary:")
print("  Patient Records: ssn, email, diagnosis_code, diagnosis_desc, prescription")
print("  Billing Records: credit_card_number, cvv, bank_account_number")

In [ ]:
# Step 6c: Verify masking is active - query data to see protection in action
sample = session.sql("""
    SELECT patient_id, first_name, last_name, ssn, email, diagnosis_code, prescription
    FROM AI_SECURITY_DEMO.HEALTHCARE.PATIENT_RECORDS
    LIMIT 5
""").to_pandas()

print("Sample data with masking policies applied:")
print("(Current role: ACCOUNTADMIN - sees unmasked data)")
print("(AI agents without ACCOUNTADMIN would see masked values)")
print()
print(sample.to_string(index=False))
print()
print("What AI agents see (non-privileged roles):")
print("  SSN:        ***-**-6789")
print("  Email:      jo****@email.com")
print("  Diagnosis:  [REDACTED - PHI]")
print("  CC:         ****-****-****-9012")

---
## Step 7: Remediation — Least Privilege for Cortex Search

Transfer ownership of Cortex Search Services from privileged roles (ACCOUNTADMIN) to dedicated service roles. This follows the **principle of least privilege** — each AI service should only have the minimum permissions it needs.

In [ ]:
# Step 7: Create least-privilege roles for Cortex Search services
roles = [
    ("CORTEX_SEARCH_BANKING_ROLE", "Least-privilege role for Banking Cortex Search services"),
    ("CORTEX_SEARCH_SUPPORT_ROLE", "Least-privilege role for Support Cortex Search services"),
    ("CORTEX_SEARCH_CLINICAL_ROLE", "Least-privilege role for Clinical Cortex Search services"),
]

print("Creating dedicated Cortex Search service roles...")
for role_name, comment in roles:
    session.sql(f"CREATE ROLE IF NOT EXISTS {role_name} COMMENT = '{comment}'").collect()
    session.sql(f"GRANT ROLE {role_name} TO ROLE SYSADMIN").collect()
    print(f"  Created: {role_name}")

# Show the services that need ownership transfer (from scanner findings)
print("\nCortex Search services flagged by AI Security scanner:")
flagged_services = session.sql("""
    SELECT 
        f.value:entity_name::VARCHAR AS SERVICE_NAME,
        'ACCOUNTADMIN (over-privileged)' AS CURRENT_OWNER,
        CASE 
            WHEN f.value:entity_name::VARCHAR ILIKE '%BANKING%' THEN 'CORTEX_SEARCH_BANKING_ROLE'
            WHEN f.value:entity_name::VARCHAR ILIKE '%SUPPORT%' THEN 'CORTEX_SEARCH_SUPPORT_ROLE'
            ELSE 'CORTEX_SEARCH_CLINICAL_ROLE'
        END AS RECOMMENDED_OWNER
    FROM snowflake.trust_center.findings,
    LATERAL FLATTEN(input => AT_RISK_ENTITIES) f
    WHERE SCANNER_ID = 'AI_SECURITY_CORTEX_SEARCH_SERVICE_PRIVILEGED_ROLES'
      AND COMPLETION_STATUS = 'SUCCEEDED'
      AND TOTAL_AT_RISK_COUNT > 0
    ORDER BY END_TIMESTAMP DESC, f.value:entity_name::VARCHAR
    LIMIT 10
""").to_pandas()

if len(flagged_services) > 0:
    print(flagged_services.to_string(index=False))
    print(f"\nTo remediate, run:")
    print(f"  GRANT OWNERSHIP ON CORTEX SEARCH SERVICE <service> TO ROLE <dedicated_role>;")
else:
    print("  No services currently flagged.")

---
## Step 8: Continuous Monitoring & Notifications

Configure the AI Security scanner package for **automated alerting** when new HIGH or CRITICAL findings are detected. Set up the monitoring query for dashboards.

In [ ]:
# Step 8a: Configure notifications for all AI Security scanners
notification_config = '{"NOTIFY_ADMINS":"TRUE","SEVERITY_THRESHOLD":"HIGH","USERS":[]}'

scanners_to_notify = [
    'AI_SECURITY_AGENT_SENSITIVE_DATA_ACCESS',
    'AI_SECURITY_CORTEX_SEARCH_SERVICE_PRIVILEGED_ROLES',
    'AI_SECURITY_CORTEX_CODE_USAGE_WITH_PAT',
    'AI_SECURITY_ADVANCED_PROMPT_INJECTION_GUARDRAIL'
]

print("Configuring notifications for AI Security scanners...")
for scanner_id in scanners_to_notify:
    session.sql(f"""
        CALL snowflake.trust_center.set_configuration(
            'NOTIFICATION', '{notification_config}', 'AI_SECURITY', '{scanner_id}'
        )
    """).collect()
    print(f"  Notifications ON: {scanner_id}")

print(f"\nAll 4 scanners now notify admins on HIGH+ severity findings")

In [ ]:
# Step 8b: Production monitoring query - use in dashboards or scheduled reports
posture_report = session.sql("""
    SELECT 
        SCANNER_SHORT_DESCRIPTION AS FINDING,
        SEVERITY,
        TOTAL_AT_RISK_COUNT AS RISK_COUNT,
        CASE 
            WHEN TOTAL_AT_RISK_COUNT = 0 THEN 'COMPLIANT'
            WHEN TOTAL_AT_RISK_COUNT <= 3 THEN 'NEEDS ATTENTION'
            ELSE 'CRITICAL ACTION REQUIRED'
        END AS POSTURE_STATUS,
        END_TIMESTAMP AS LAST_SCANNED
    FROM snowflake.trust_center.findings
    WHERE SCANNER_PACKAGE_ID = 'AI_SECURITY'
      AND COMPLETION_STATUS = 'SUCCEEDED'
      AND END_TIMESTAMP = (
          SELECT MAX(END_TIMESTAMP) 
          FROM snowflake.trust_center.findings f2 
          WHERE f2.SCANNER_ID = findings.SCANNER_ID 
            AND f2.COMPLETION_STATUS = 'SUCCEEDED'
      )
    ORDER BY TOTAL_AT_RISK_COUNT DESC
""").to_pandas()

print(f"{'='*80}")
print("CURRENT AI SECURITY POSTURE REPORT")
print(f"{'='*80}")
print(posture_report.to_string(index=False))

---
## Step 9: Executive Security Dashboard

Aggregate all AI Security metrics into an executive-ready visualization with compliance scoring.

In [ ]:
# Step 9: Executive Dashboard
import altair as alt
import pandas as pd
alt.renderers.enable('mimetype')

# Get latest findings
dashboard_data = session.sql("""
    SELECT 
        SCANNER_SHORT_DESCRIPTION AS scanner,
        TOTAL_AT_RISK_COUNT AS violations,
        CASE TOTAL_AT_RISK_COUNT WHEN 0 THEN 'Compliant' ELSE 'Violation' END AS status
    FROM snowflake.trust_center.findings
    WHERE SCANNER_PACKAGE_ID = 'AI_SECURITY'
      AND COMPLETION_STATUS = 'SUCCEEDED'
    ORDER BY END_TIMESTAMP DESC
    LIMIT 4
""").to_pandas()

# Compliance donut chart
compliance_data = dashboard_data.groupby('status').size().reset_index(name='count')

pie = alt.Chart(compliance_data).mark_arc(innerRadius=50).encode(
    theta='count:Q',
    color=alt.Color('status:N', 
                    scale=alt.Scale(domain=['Compliant', 'Violation'], 
                                   range=['#27ae60', '#e74c3c']),
                    title='Status'),
    tooltip=['status:N', 'count:Q']
).properties(
    title='AI Security Compliance',
    width=250,
    height=250
)

# Violations bar chart
bars = alt.Chart(dashboard_data).mark_bar().encode(
    x=alt.X('violations:Q', title='At-Risk Entities'),
    y=alt.Y('scanner:N', title='', sort='-x'),
    color=alt.Color('status:N',
                    scale=alt.Scale(domain=['Compliant', 'Violation'],
                                   range=['#27ae60', '#e74c3c']))
).properties(
    title='Violations by Scanner',
    width=400,
    height=200
)

display(pie | bars)

# Executive summary metrics
total_violations = dashboard_data['violations'].sum()
compliant_count = len(dashboard_data[dashboard_data['violations'] == 0])
total_scanners = len(dashboard_data)
compliance_pct = (compliant_count / total_scanners) * 100

print(f"\n{'='*60}")
print(f"  AI SECURITY POSTURE - EXECUTIVE SUMMARY")
print(f"{'='*60}")
print(f"  Scanners Active:      {total_scanners}/4")
print(f"  Scanners Compliant:   {compliant_count}/{total_scanners} ({compliance_pct:.0f}%)")
print(f"  Total Violations:     {total_violations}")
print(f"  Risk Level:           {'LOW' if total_violations == 0 else 'HIGH' if total_violations > 5 else 'MEDIUM'}")
print(f"  Guardrails:           ENABLED (prompt injection protection)")
print(f"  Masking Policies:     6 policies on 8 columns")
print(f"  Data Classification:  10 columns tagged")
print(f"  Notifications:        HIGH+ severity alerts to admins")
print(f"{'='*60}")

In [ ]:
# Step 9b: Trust Center cost monitoring
cost_data = session.sql("""
    SELECT
        USAGE_DATE AS DATE,
        CREDITS_USED AS CREDITS
    FROM snowflake.account_usage.metering_daily_history
    WHERE service_type = 'TRUST_CENTER'
      AND USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE())
    ORDER BY USAGE_DATE DESC
""").to_pandas()

if len(cost_data) > 0:
    total_cost = cost_data['CREDITS'].sum()
    print(f"Trust Center cost (last 30 days): {total_cost:.4f} credits")
    
    cost_chart = alt.Chart(cost_data).mark_area(
        color='#3498db', opacity=0.6
    ).encode(
        x=alt.X('DATE:T', title='Date'),
        y=alt.Y('CREDITS:Q', title='Credits Used')
    ).properties(
        title='Trust Center Daily Credit Usage (Last 30 Days)',
        width=600,
        height=150
    )
    display(cost_chart)
else:
    print("No Trust Center cost data available yet (metrics appear after 24h)")

---
## Step 10: Key Takeaways & Enterprise Best Practices

### What We Demonstrated

| Capability | Implementation | Business Value |
|-----------|----------------|----------------|
| AI Security Scanner Package | Enabled 4 scanners, on-demand execution | Continuous monitoring of AI workload security |
| Data Classification | Tagged 10 columns with PRIVACY_CATEGORY | Machine-readable data sensitivity inventory |
| Cortex AI Guardrails | Enabled prompt injection protection | Runtime defense against adversarial AI attacks |
| Dynamic Data Masking | 6 policies applied to 8 columns | PII/PHI protection for AI agent access (HIPAA/PCI) |
| Least Privilege | Dedicated roles for Cortex Search | Prevent privilege escalation through AI services |
| Continuous Monitoring | Notifications + daily scans | Automated alerting on new security violations |

### The 4 AI Security Scanners

| Scanner | Threat Addressed | Remediation |
|---------|-----------------|-------------|
| Cortex Search Privileged Roles | Over-privileged AI services | Transfer ownership to dedicated role |
| Cortex Code PAT Usage | Unrestricted developer access | Set ALLOWED_ROLES on PATs + network policy |
| Sensitive Data Accessed by Agent | Unmasked PII exposure to AI | Apply masking + row-access policies |
| Cortex AI Guardrails | Prompt injection / jailbreaks | `ALTER ACCOUNT SET AI_SETTINGS` |

### Enterprise Recommendations

1. **Enable AI Security scanner package** in ALL accounts using Cortex AI features
2. **Classify all sensitive data** using `SNOWFLAKE.CORE.PRIVACY_CATEGORY` tags
3. **Apply masking policies** BEFORE granting AI agents access to sensitive tables
4. **Enable Cortex AI Guardrails** for runtime prompt injection protection
5. **Transfer Cortex Search ownership** from admin roles to dedicated service roles
6. **Configure notifications** for automated alerting on new findings
7. **Monitor Trust Center costs** via `metering_daily_history` view
8. **Review AI Security tab weekly** in Snowsight (Governance > Trust Center > AI Security)

### Access the AI Security Tab in Snowsight

```
Navigate to: Snowsight > Governance & Security > Trust Center > AI Security
```

This provides:
- AI Agents inventory (count of all Cortex Agents)
- Scanner coverage status (4/4 enabled)
- Guardrails configuration status
- Violations and detections trend charts (7-day view)

### Related Snowflake Summit 2026 Security Features

| Feature | Description |
|---------|-------------|
| **Agent Identity** | Each AI agent gets a verified identity and full audit trail |
| **Data Exfiltration Policies** | Prevent unauthorized data export from AI workloads |
| **Multi-Party Authorization** | Require multiple approvals for sensitive AI operations |
| **Cortex AI Guardrails (GA)** | ML-driven prompt injection and jailbreak detection |
| **Model-Level RBAC** | Role-based access control for individual AI models |

---
**Demo Complete.** Your account now has AI Security Posture Management fully operational with continuous monitoring, guardrails, data protection, and automated alerting.

---
## Post-Demonstration: Disable AI Security Scanners (Cost Control)

Running the AI Security scanner package incurs **serverless compute credits** for each scheduled scan. After completing the demonstration, disable the scanners to stop continuous monitoring costs.

> **Note:** The Cortex AI Guardrails (`AI_SETTINGS`) remain active — that is a runtime protection layer and does not incur Trust Center scanning costs.

In [ ]:
# Post-Demo: Disable AI Security scanner package to stop incurring costs
print("Disabling AI Security scanner package and all scanners...")

# Disable the package (stops all scheduled scans)
session.sql("CALL snowflake.trust_center.set_configuration('ENABLED', 'FALSE', 'AI_SECURITY', false)").collect()

# Explicitly disable each individual scanner
scanners_to_disable = [
    'AI_SECURITY_AGENT_SENSITIVE_DATA_ACCESS',
    'AI_SECURITY_CORTEX_SEARCH_SERVICE_PRIVILEGED_ROLES',
    'AI_SECURITY_ADVANCED_PROMPT_INJECTION_GUARDRAIL',
    'AI_SECURITY_CORTEX_CODE_USAGE_WITH_PAT'
]

for scanner_id in scanners_to_disable:
    session.sql(f"CALL snowflake.trust_center.set_configuration('ENABLED', 'FALSE', 'AI_SECURITY', '{scanner_id}')").collect()
    print(f"  Disabled: {scanner_id}")

# Verify everything is off
status = session.sql("""
    SELECT 
        sp.NAME AS PACKAGE, sp.STATE AS PKG_STATE,
        s.ID AS SCANNER_ID, s.STATE AS SCANNER_STATE
    FROM snowflake.trust_center.scanners s
    JOIN snowflake.trust_center.scanner_packages sp ON s.SCANNER_PACKAGE_ID = sp.ID
    WHERE s.SCANNER_PACKAGE_ID = 'AI_SECURITY'
""").to_pandas()

print(f"\n{'='*60}")
print("VERIFICATION: All AI Security scanners disabled")
print(f"{'='*60}")
print(status.to_string(index=False))
print(f"\nNo further Trust Center credits will be consumed for AI Security scanning.")
print("To re-enable in production, run Step 3 of this notebook.")